In [2]:
import numpy as np
from scipy.stats import norm

# Q1

(a) Compute the implied volatility of the option using the Black-Scholes Model. Use 
trial-and-error or interpolation. Try at least three volatility levels in the range 
0.1 to 0.5. Use the Black-Scholes formula for a European call option.

In [3]:
def current_value_call(S0, K, r, T, sig):
    # sig is for sigma, the volatility
    d1 = np.log(S0/K) + (r + (sig**2)/2)*T
    d1 /= sig*np.sqrt(T)
    d2 = d1 - sig*np.sqrt(T)

    N1 = norm.cdf(d1, loc=0, scale=1)
    N2 = norm.cdf(d2, loc=0, scale=1)

    C = S0*N1 - K*np.exp(-1*r*T)*N2
    return C

In [4]:
# Calculating IV using quadratic interpolation
C = 4.20 # Curent Value of the Option
S0 = 38.0 # Current Stock Price
K = 35 # Strike Price
T = 4.0/12.0 # Time to expiry in years
r = 0.06 # risk-free rate

# Fitting a degree 2 polynomial over these data points and their corresponding current values of European Call
proposed_volatilities = np.array([0.1, 0.3, 0.5])
current_values = current_value_call(S0, K, r, T, proposed_volatilities)
coeffs = np.polyfit(current_values, proposed_volatilities, 2)

# IV will be the value of this fitted curve at C = 4.2
IV = np.polyval(coeffs, C)
print(f"Implied Volatility for the given stock is : {IV:.3f}")

Implied Volatility for the given stock is : 0.198


(b) Suppose the implied volatility is accepted to be 0.28 from market estimation.
Now, using this volatility, calculate the price of a European put option with the
same strike and maturity. Use the Black-Scholes formula for a European put
option or apply put-call parity.

In [5]:
def current_value_put(S0, K, r, T, sig):
    d1 = np.log(S0/K) + (r + (sig**2)/2)*T
    d1 /= sig*np.sqrt(T)
    d2 = d1 - sig*np.sqrt(T)

    N1 = norm.cdf(-d1, loc=0, scale=1)
    N2 = norm.cdf(-d2, loc=0, scale=1)

    P = -S0*N1 + K*np.exp(-1*r*T)*N2
    return P

In [6]:
volatility = 0.28

# Put Call Parity
# C - P = S - K.exp(-rT)

P = current_value_put(S0, K, r, T, volatility)
print(f"Price of Put Option for given stock and volatility : ${P:.3f}")

Price of Put Option for given stock and volatility : $0.932


A firm owns a patented drug that can be commercialized within the next 4
months. The launch involves a fixed regulatory and setup cost equivalent to
$35 million. The expected net revenue from the product launch (treated as the
“stock”) is currently estimated at $38 million and follows a log-normal diffusion
process. There is no intermediate revenue, and the firm can only make the launch
decision now. Should the firm launch the product? Justify using a real options
perspective and your results from (a) and (b). Assume the launch opportunity
resembles a European call option.

In [7]:
# The real options approach justifies the launch only if the option value of the launch exceeds the payoff at launch.
value = current_value_call(S0, K, r, T, volatility)

if (value >= S0 - K):
    print(f"Firm should launch as Current Value(${value:.2f} million) >= S-K(${3} million)")
else:
    print(f"Firm should not launch as Current Value(${value:.2f} million) <= S-K(${3} million)")

Firm should launch as Current Value($4.62 million) >= S-K($3 million)
